# Planta baixa a partir da nuvem de pontos

**Projeto IPE 2 — Seção de Engenharia Cartográfica do IME**

**Objetivos**

1. Nivelar a nuvem registrada e isolar o pavimento
2. Separar parede de mobiliário por **persistência vertical**
3. Vetorizar as faces e medir a espessura de cada parede
4. Detectar e classificar portas e janelas no plano da parede
5. Gravar a planta em DXF por camadas e auditar o resultado

Este notebook narra o que o pacote `plantabaixa/` faz, etapa por etapa, mostrando
a evidência em que cada decisão se apoia. Para rodar o resultado final de uma vez
só, use a linha de comando:

```powershell
python .\planta_baixa.py ..\SN_000055_certo -o planta_secao_cartografica.dxf --diagnosticos diag\
```

## 1. De onde vem a nuvem

O levantamento bruto do SLAM 100 (`SN_00055/PROJ1`) tem o `.pcap` do Pandar
XT-16, o encoder da 2ª plataforma giratória e a IMU — mas **não** uma nuvem
registrada. A nuvem vem do SLAM GO POST, exportada em `.las`/`.laz`.

A célula abaixo usa a nuvem real se ela já estiver na pasta e, enquanto não
estiver, cai na nuvem sintética de planta conhecida que serve de gabarito aos
testes. Todo o resto do notebook roda igual nos dois casos.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))

from plantabaixa import nuvem as mod_nuvem
from plantabaixa import paredes as mod_paredes
from plantabaixa import vaos as mod_vaos
from plantabaixa import dxf as mod_dxf
from plantabaixa import qc as mod_qc

CANDIDATAS = [Path("../SN_000055_certo"), Path("SN_000055_certo"),
              Path("LevantamentoLidarIME/SN_000055_certo")]
entrada = next((p for p in CANDIDATAS if p.exists()), None)

if entrada is not None:
    print(f"Nuvem real: {mod_nuvem.descobrir_nuvem(entrada)}")
    nuvem = mod_nuvem.carregar_nuvem(entrada, voxel=0.02)
else:
    print("SN_000055_certo ainda nao esta na pasta; usando a nuvem sintetica de gabarito.")
    from tests import sintetico
    centros, pesos = mod_nuvem.voxelizar(sintetico.gerar(), 0.02)
    nuvem = mod_nuvem.preparar_nuvem(centros, pesos, voxel=0.02)

print(f"{len(nuvem.xyz):,} voxels de {nuvem.voxel * 100:.0f} cm".replace(",", "."))

## 2. Nivelamento: por que ele não é opcional

Tudo o que vem depois supõe que `z` é a vertical verdadeira e que o piso está em
`z = 0`. Um desaprumo de **1 grau** desloca o teto em **50 cm** numa sala de
30 m — e o corte por altura perde o sentido.

O piso é ajustado por RANSAC no **quartil inferior de altura**, e não no pico do
histograma. Com a nuvem desaprumada o pico do piso se esborracha ao longo da
sala, mas o quartil inferior continua dominado por pontos de piso. Note também
que, naquele quartil, pé de parede e pé de móvel costumam ser *mais numerosos*
que o próprio piso: por isso o vínculo de normal quase vertical no RANSAC é o que
impede uma parede próxima de roubar o ajuste.

In [ ]:
print(f"Desaprumo corrigido: {nuvem.desnivel_graus:.3f} graus")
print(f"Altura piso-teto...: {nuvem.z_teto:.3f} m")
for aviso in nuvem.avisos:
    print(f"AVISO: {aviso}")

figura, eixo = plt.subplots(figsize=(9, 3.5))
eixo.hist(nuvem.xyz[:, 2], bins=140, weights=nuvem.pesos, color="tab:blue")
eixo.axvline(0.0, color="tab:green", lw=1.5, label="piso (z = 0)")
eixo.axvline(nuvem.z_teto, color="tab:red", lw=1.5, label=f"teto ({nuvem.z_teto:.2f} m)")
eixo.set_xlabel("altura (m)")
eixo.set_ylabel("pontos")
eixo.set_title("Distribuicao de altura depois do nivelamento")
eixo.legend()
plt.show()

## 3. O problema central: armário também é vertical

O caminho óbvio para achar parede é cortar a nuvem numa faixa de altura — de 1,0
a 2,0 m, digamos — e olhar o que sobra em planta. **E é justamente aí que o
método quebra em ambiente mobiliado**: nesse corte, um armário de 2 m é
indistinguível de uma parede. Armários encostam em paredes, e o resultado é
parede grossa demais onde há armário e parede inventada onde há estante.

O que separa os dois não é *onde* estão, e sim **quanto do vão piso-teto cada
coluna vertical ocupa** e **se ela encosta no teto**. Parede atravessa o vão
inteiro; armário para antes.

A célula abaixo aplica os dois critérios lado a lado.

In [ ]:
grade = mod_paredes.mapa_ocupacao(nuvem.xyz, nuvem.z_teto, res=0.02, theta=0.0)

indices = np.floor((nuvem.xyz[:, :2] - grade.origem) / grade.res).astype(np.int64)
na_faixa = (nuvem.xyz[:, 2] >= 1.0) & (nuvem.xyz[:, 2] <= 2.0)
faixa_fixa = np.zeros(grade.forma, dtype=bool)
faixa_fixa[indices[na_faixa, 1], indices[na_faixa, 0]] = True

persistencia = grade.mascara(0.50)

figura, eixos = plt.subplots(1, 2, figsize=(14, 6))
eixos[0].imshow(faixa_fixa, origin="lower", cmap="Greys", interpolation="nearest")
eixos[0].set_title(f"Corte fixo de 1,0 a 2,0 m: {faixa_fixa.sum()} celulas")
eixos[1].imshow(persistencia, origin="lower", cmap="Greys", interpolation="nearest")
eixos[1].set_title(f"Persistencia >= 0,50 e encosta no teto: {persistencia.sum()} celulas")
for eixo in eixos:
    eixo.set_xticks([])
    eixo.set_yticks([])
plt.tight_layout()
plt.show()

print(f"O corte fixo marca {faixa_fixa.sum() / max(1, persistencia.sum()):.1f}x mais celulas;")
print("a diferenca e mobiliario alto, que o criterio de encostar no teto descarta.")

A medida em si é um mapa contínuo. Vale olhar o histograma dela antes de fixar o
limiar: parede e mobiliário aparecem como populações separadas, e o
`--persistencia` deve cair no vale entre as duas.

In [ ]:
figura, eixos = plt.subplots(1, 2, figsize=(14, 5))
imagem = eixos[0].imshow(grade.score, origin="lower", cmap="magma", vmin=0, vmax=1,
                         interpolation="nearest")
figura.colorbar(imagem, ax=eixos[0], label="fracao do vao piso-teto ocupada")
eixos[0].set_title("Persistencia vertical por celula")
eixos[0].set_xticks([])
eixos[0].set_yticks([])

valores = grade.score[grade.score > 0]
eixos[1].hist(valores, bins=50, color="tab:purple")
eixos[1].axvline(0.50, color="tab:red", lw=1.5, label="--persistencia 0,50")
eixos[1].set_yscale("log")
eixos[1].set_xlabel("score")
eixos[1].set_ylabel("celulas (log)")
eixos[1].set_title("Onde cai o limiar")
eixos[1].legend()
plt.tight_layout()
plt.show()

## 4. Vetorização: dar a cada eixo só as células dele

Com a máscara pronta, o passo natural é projetar as células num eixo e procurar
picos — cada pico seria uma face de parede. Feito assim, não funciona: uma parede
de 16 m que corre em *x* espalha 800 células ao longo do eixo *y*, e esse fundo é
alto o bastante para fabricar picos onde não existe parede alguma. Na primeira
versão deste algoritmo, 8 paredes viraram 58.

A correção é classificar cada célula pela **orientação local da estrutura** a que
pertence, via tensor de estrutura, antes de histogramar. Cada eixo recebe então
apenas as suas, e o fundo desaparece. As células de cruzamento, onde não há
direção dominante, ficam de fora — e é por isso que as pontas das paredes chegam
aparadas e precisam, depois, ser estendidas até as perpendiculares.

In [ ]:
angulo, coerencia = mod_paredes.orientacao_local(persistencia, grade.res)
rotulos = mod_paredes.classificar_celulas(angulo, coerencia, persistencia)

nomes = {0: "eixo x", 1: "eixo y", 2: "obliqua", -1: "descartada"}
for valor, quantas in zip(*np.unique(rotulos[persistencia], return_counts=True)):
    print(f"  {nomes[int(valor)]:11}: {quantas:6d} celulas")

mapa = np.full(rotulos.shape, np.nan)
for valor in (0, 1, 2):
    mapa[rotulos == valor] = valor
figura, eixo = plt.subplots(figsize=(9, 7))
eixo.imshow(mapa, origin="lower", cmap="viridis", interpolation="nearest")
eixo.set_title("Celulas classificadas pela orientacao local")
eixo.set_xticks([])
eixo.set_yticks([])
plt.show()

Agora a extração completa: densidade suavizada dos deslocamentos em cada eixo →
faces; faces paralelas a 8–35 cm com extensões sobrepostas → parede, com a
**espessura medida** entre elas; pontas estendidas até a face distante da parede
perpendicular → cantos maciços.

In [ ]:
extracao = mod_paredes.extrair_paredes(nuvem.xyz, nuvem.z_teto, res=0.02, persistencia=0.50)
print(f"{len(extracao.faces)} faces -> {len(extracao.paredes)} paredes; "
      f"giro de esquadro {np.degrees(extracao.theta):.3f} graus\n")
print(f"{'parede':7} {'dir':4} {'compr':>7} {'espess':>8} {'faces':>6} {'residuo':>9}")
for parede in extracao.paredes:
    direcao = "x" if abs(parede.direcao[0]) > 0.5 else "y"
    residuo = np.sqrt(np.mean([f.residuo ** 2 for f in parede.faces])) * 1000
    print(f"{parede.nome:7} {direcao:4} {parede.comprimento:6.2f}m "
          f"{parede.espessura * 100:6.1f}cm {len(parede.faces):6d} {residuo:7.1f}mm")
for aviso in extracao.avisos:
    print(f"\nAVISO: {aviso}")

## 5. Portas e janelas: o plano da parede, não a planta

Procurar vão como lacuna na máscara 2D parece natural e **não funciona para
janela**. A coluna vertical sobre uma janela tem alvenaria embaixo (do piso ao
peitoril) e em cima (da verga ao teto): persistência em torno de 0,5 —
exatamente o limiar que separa parede de móvel. A janela cairia de um lado ou do
outro ao sabor de um centímetro de peitoril.

O lugar sem ambiguidade é o plano da própria parede. Projetando os pontos em
`(s, h)` — abscissa ao longo do eixo e altura —, a alvenaria é região cheia e
cada abertura é um retângulo vazio.

Falta então separar **abertura** de **oclusão**: os dois são vazio no mapa. A
evidência vem da física do levantamento. Dentro da espessura de uma parede maciça
não há nada para o sensor ver; se existem pontos ali, é porque há abertura — e a
altura deles diz qual:

| Pontos dentro da pegada da parede | Leitura |
|---|---|
| piso atravessando, base junto ao chão | **porta** (ou passagem, se larga) |
| peitoril entre 0,4 e 1,5 m | **janela** |
| nada | **oclusão** — trecho não levantado |

In [ ]:
vaos, analises, avisos = mod_vaos.detectar_vaos(
    extracao.paredes, extracao.xyz_predio, nuvem.z_teto)

print(f"{'parede':7} {'classe':15} {'largura':>8} {'altura':>8} {'base':>7} {'nucleo':>7}")
for vao in vaos:
    print(f"{vao.parede:7} {vao.classe:15} {vao.largura:7.2f}m {vao.altura:7.2f}m "
          f"{vao.base:6.2f}m {vao.pontos_nucleo:7d}")
for aviso in avisos:
    print(f"AVISO: {aviso}")

O mapa `(s, h)` de uma parede mostra a decisão inteira num quadro só: onde há
alvenaria, onde está o retângulo vazio, e qual classe ele recebeu — com o número
de pontos dentro da pegada, que foi o que decidiu.

In [ ]:
com_vaos = sorted(extracao.paredes,
                  key=lambda p: -sum(1 for v in vaos if v.parede == p.nome))[:3]
for parede in com_vaos:
    analise = analises[parede.nome]
    if analise.material.size <= 1:
        continue
    figura, eixo = plt.subplots(figsize=(11, 3))
    extensao = [analise.s0, analise.s0 + analise.material.shape[0] * analise.celula,
                analise.h0, analise.h0 + analise.material.shape[1] * analise.celula]
    eixo.imshow(analise.material.T, origin="lower", extent=extensao, cmap="Greys",
                aspect="auto", interpolation="nearest")
    for vao in analise.vaos:
        cor = {"porta": "tab:green", "passagem": "tab:green",
               "janela": "tab:blue"}.get(vao.classe, "tab:orange")
        eixo.add_patch(plt.Rectangle((vao.inicio, vao.base), vao.largura, vao.altura,
                                     fill=False, edgecolor=cor, lw=2.2))
        eixo.text(vao.centro, vao.topo + 0.06, f"{vao.classe} ({vao.pontos_nucleo} pts)",
                  color=cor, ha="center", fontsize=9)
    eixo.set_xlabel("s ao longo da parede (m)")
    eixo.set_ylabel("altura (m)")
    eixo.set_title(f"{parede.nome}: alvenaria no plano da parede")
    plt.tight_layout()
    plt.show()

## 6. O desenho

O corte horizontal da planta passa a **1,40 m**, e só as aberturas que o corte
atravessa interrompem a alvenaria. Um basculante que começa a 1,80 m é
representado tracejado **sem** seccionar a parede — que é como um projetista o
desenha. A oclusão também não secciona: aquele trecho tem parede, só não foi
visto.

In [ ]:
caminho = mod_dxf.escrever(
    "saida/planta_notebook.dxf", extracao.paredes, vaos, extracao.theta,
    notas=[f"Gerado do notebook; teto a {nuvem.z_teto:.2f} m"], sobrescrever=True)
print(f"DXF: {caminho}")
for camada, quantas in sorted(mod_dxf.conferir(caminho).items()):
    print(f"  {camada:12}: {quantas:4d} entidades")

mod_qc.previa_planta(extracao, vaos, "saida/previa_notebook.png")
figura, eixo = plt.subplots(figsize=(13, 9))
eixo.imshow(plt.imread("saida/previa_notebook.png"))
eixo.axis("off")
plt.show()

## 7. Controle de qualidade

O relatório é o que permite confiar — ou desconfiar — do desenho. O resíduo por
parede diz quão plana é cada face; a lista de avisos diz o que **não** foi
medido.

In [ ]:
relatorio = mod_qc.montar_relatorio(nuvem, extracao, vaos, avisos)
print(relatorio.texto())

## 8. O que este algoritmo não faz

- **Porta fechada no levantamento é indetectável.** A folha é superfície plana e
  lê como alvenaria. O algoritmo reporta como *suspeita* o trecho de face recuado
  de 2 a 9 cm com largura de porta — indício, nunca detecção.
- **Trecho ocluído não é abertura**, e a parede é desenhada contínua por cima
  dele. Mas aquele trecho não foi medido, e isso consta do relatório.
- **O lado para o qual a porta abre é hipótese de desenho**, adotada pelo lado com
  mais piso levantado. A nuvem não carrega essa informação.
- **Um pavimento por execução.** Massa de pontos acima do teto é recortada e
  avisada.
- **A grade precisa resolver a parede mais fina**: cerca de 3 células na
  espessura. Abaixo disso a espessura vira artefato da resolução, e o relatório
  avisa em vez de mentir.

### Próximos passos possíveis

Ambientes com área e rótulo (o espaço livre já é calculado internamente, para
saber de que lado está o ambiente), cotas automáticas e prancha plotada em PDF,
exportação para GeoJSON e múltiplos pavimentos.